# Figure S1: the reported within-participant case study
Fixed manuscript pair: `UFALL_PD_0005`, bouts 61 and 38 (the manuscript abbreviates the ID as UFALL_PD_005). No search for an alternative pair is performed. The original environmental plot normalises each bout across its five metrics; this visual scaling is retained and is not a population standardisation. Raw video/mask panels require the external source data.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
OUT = Path("results")
OUT.mkdir(exist_ok=True)
import matplotlib.pyplot as plt
from helpers.paper_figures import show_video_frames

bouts = pd.read_parquet('derived/bouts.parquet')
GAIT_COLS = ['BoutStepTCV','BoutStrideTCV','BoutSwingTCV','BoutStanceTCV']
ENV_COLS = ['EnvClutterIndexMean','EnvPathObstructionIndexMean','EnvVisibleOpenFloorRatioMean','EnvCentralCorridorOccupancyMean','EnvTightCorridorIndexMean']
pair = bouts.loc[bouts.Participant.eq('UFALL_PD_0005') & bouts.BoutIndex.isin([61,38])].set_index('BoutIndex')
assert set(pair.index)=={61,38}, 'Missing manuscript case-study bouts'

hard_row=pair.loc[61];easy_row=pair.loc[38]
sample_compare=pd.DataFrame({'Metric':GAIT_COLS+ENV_COLS,
    'HighVAR_HardContext':hard_row[GAIT_COLS+ENV_COLS].to_numpy(float),
    'HighVAR_EasyContext':easy_row[GAIT_COLS+ENV_COLS].to_numpy(float)})

display(sample_compare)
sample_compare.to_csv(OUT/'figureS1_metrics.csv',index=False)

In [ ]:
hard_color = '#1f4e79'
easy_color = '#4fa3d1'
line_color = '#9e9e9e'


# ==========================================================================
# FIGURE 1 — GAIT
# ==========================================================================

fig, ax = plt.subplots(figsize=(7, 5))

gait_plot = sample_compare[
    sample_compare['Metric'].isin(GAIT_COLS)
].copy()

gait_labels = {
    'BoutStepTCV': 'Step time CV',
    'BoutStanceTCV': 'Stance time CV',
    'BoutSwingTCV': 'Swing time CV',
    'BoutStrideTCV': 'Stride time CV',
}

y = np.arange(len(gait_plot))

hard_vals = gait_plot['HighVAR_HardContext'].to_numpy(dtype=float)
easy_vals = gait_plot['HighVAR_EasyContext'].to_numpy(dtype=float)

# Connecting lines
for yi, h, e in zip(y, hard_vals, easy_vals):
    ax.plot(
        [h, e],
        [yi, yi],
        color=line_color,
        linewidth=2,
        zorder=1
    )

# Dots
ax.scatter(
    hard_vals,
    y,
    s=90,
    color=hard_color,
    label="Bout 61",
    zorder=3
)

ax.scatter(
    easy_vals,
    y,
    s=90,
    color=easy_color,
    label="Bout 38",
    zorder=3
)

ax.set_yticks(y)
ax.set_yticklabels([
    gait_labels.get(metric, metric)
    for metric in gait_plot['Metric']
])
ax.set_xlim([0, 0.3])

ax.invert_yaxis()

ax.set_xlabel('Coefficient of variation')

ax.grid(axis='x', alpha=0.3)
ax.grid(axis='y', visible=False)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.tick_params(axis='y', length=0)

ax.legend(
    frameon=False,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.12),
    ncol=1
)

plt.tight_layout()
fig.savefig(OUT/'figureS1_gait.png',dpi=160,bbox_inches='tight')
plt.show()


# ==========================================================================
# FIGURE 2 — ENVIRONMENT
# ==========================================================================

fig, ax = plt.subplots(figsize=(7, 5))

env_metrics_plot = [
    'EnvClutterIndexMean',
    'EnvPathObstructionIndexMean',
    'EnvVisibleOpenFloorRatioMean',
    'EnvCentralCorridorOccupancyMean',
    'EnvTightCorridorIndexMean',
]

env_labels = {
    'EnvClutterIndexMean': 'Clutter',
    'EnvPathObstructionIndexMean': 'Path obstruction',
    'EnvVisibleOpenFloorRatioMean': 'Visible open floor',
    'EnvCentralCorridorOccupancyMean': 'Corridor occupancy',
    'EnvTightCorridorIndexMean': 'Corridor constraint',
}

env_plot = sample_compare[
    sample_compare['Metric'].isin(env_metrics_plot)
].copy()

# Normalise each bout across environmental metrics
env_norm = env_plot.copy()

for col in ['HighVAR_HardContext', 'HighVAR_EasyContext']:
    vals = env_norm[col].to_numpy(dtype=float)
    mn, mx = vals.min(), vals.max()
    env_norm[col] = (vals - mn) / (mx - mn + 1e-12)

y = np.arange(len(env_norm))

hard_env = env_norm['HighVAR_HardContext'].to_numpy(dtype=float)
easy_env = env_norm['HighVAR_EasyContext'].to_numpy(dtype=float)

# Connecting lines
for yi, h, e in zip(y, hard_env, easy_env):
    ax.plot(
        [h, e],
        [yi, yi],
        color=line_color,
        linewidth=2,
        zorder=1
    )

# Dots
ax.scatter(
    hard_env,
    y,
    s=90,
    color=hard_color,
    label="Bout 61",
    zorder=3
)

ax.scatter(
    easy_env,
    y,
    s=90,
    color=easy_color,
    label="Bout 38",
    zorder=3
)

ax.set_yticks(y)
ax.set_yticklabels([
    env_labels.get(metric, metric)
    for metric in env_norm['Metric']
])

ax.invert_yaxis()

ax.set_xlim(-0.05, 1.05)
ax.set_xlabel('Normalised environmental metric')

ax.grid(axis='x', alpha=0.3)
ax.grid(axis='y', visible=False)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.tick_params(axis='y', length=0)

ax.legend(
    frameon=False,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.12),
    ncol=1
)

plt.tight_layout()
fig.savefig(OUT/'figureS1_environment.png',dpi=160,bbox_inches='tight')
plt.show()

## Representative frames from the reported bouts

In [ ]:
frames=pd.read_parquet('derived/frames.parquet',columns=['Participant','BoutIndex','FrameIndex','FrameNumber','RawMaskPath'])
examples=[]
for bout_index in [61,38]:
    f=frames.loc[frames.Participant.eq('UFALL_PD_0005') & frames.BoutIndex.eq(bout_index)].dropna(subset=['FrameNumber']).sort_values('FrameIndex')
    if len(f): examples.append(f.iloc[np.linspace(0,len(f)-1,3,dtype=int)])
if examples:
    selected_frames=pd.concat(examples)
    selected_frames.to_csv(OUT/'figureS1_frames.csv',index=False)
    show_video_frames(selected_frames,OUT/'figureS1_context.png')